In [0]:
%sql
-- Se agrega campo date_product a las tablas
ALTER TABLE workspace.default.products_2000000
ADD COLUMNS (date_product DATE);

ALTER TABLE workspace.default.products_200000_cluster
ADD COLUMNS (date_product DATE);

-- Se agreva valor aletorio
UPDATE workspace.default.products_2000000
SET date_product = DATE_ADD(
    DATE('2024-01-01'),
    ABS(HASH(`Internal ID`)) % DATEDIFF(DATE('2026-07-10'), DATE('2024-01-01'))
);

UPDATE workspace.default.products_200000_cluster
SET date_product = DATE_ADD(
    DATE('2024-01-01'),
    ABS(HASH(`Id`)) % DATEDIFF(DATE('2026-07-10'), DATE('2024-01-01'))
);

 

In [0]:
# BAJAR EL TAMAÑO DE LOS ARCHIVOS
spark.conf.set("spark.sql.files.maxRecordsPerFile", 10000)

spark.conf.set(
    "spark.databricks.delta.targetFileSize",
    "10485760"
)

In [0]:
# CREAR TABLA REPARTICIONADA
df = spark.table("workspace.default.products_200000_cluster")

(df.repartition(100)
 .write
 .format("delta")
 .mode("overwrite")
 .saveAsTable("productos"))



In [0]:
%sql
select * from workspace.default.productos;

ALTER TABLE workspace.default.productos
CLUSTER BY (date_product, Category);

OPTIMIZE workspace.default.productos FULL;


EXPLAIN FORMATTED
SELECT * FROM workspace.default.productos
WHERE date_product = '2024-06-22'
AND Category LIKE 'Home & Kitchen';

DESCRIBE DETAIL workspace.default.productos;



In [0]:
%sql
select * from workspace.default.products_2000000;

DESCRIBE DETAIL workspace.default.products_2000000;

DESCRIBE DETAIL workspace.default.products_200000_cluster;

SHOW CREATE TABLE workspace.default.products_2000000;

SHOW PARTITIONS workspace.default.products_200000_cluster;

-- Crear CLuster

ALTER TABLE workspace.default.products_200000_cluster
CLUSTER BY (date_product, category);

OPTIMIZE workspace.default.products_200000_cluster;

ALTER TABLE workspace.default.products_2000000
CLUSTER BY (date_product, category);


OPTIMIZE workspace.default.products_2000000;

DESCRIBE HISTORY workspace.default.products_200000_cluster;

ANALYZE TABLE workspace.default.products_200000_cluster
COMPUTE STATISTICS FOR ALL COLUMNS;

EXPLAIN FORMATTED
SELECT * FROM workspace.default.products_200000_cluster 
WHERE Id = 100;

EXPLAIN FORMATTED
SELECT * FROM workspace.default.products_200000_cluster
WHERE date_product = '2024-06-22'
AND Id = 100;

SELECT COUNT(*) FROM workspace.default.products_200000_cluster


In [0]:
path = (
    spark.sql("""
    DESCRIBE DETAIL workspace.default.products_200000_cluster
    """)
    .select("location")
    .collect()[0][0]
)

print(path)

In [0]:

# Ver archivos fisicos de una tabla
path = spark.sql("""
DESCRIBE DETAIL workspace.default.products_200000_cluster
""").select("location").first()[0]

print(path)


In [0]:
display(dbutils.fs.ls(path))